# Política de desconto para empresas de diferentes portes

Este notebook apresenta cinco padrões de projeto aplicados ao tema de política de desconto para empresas com diferentes tamanhos:
- Factory Method
- Observer
- Strategy
- Adapter
- Decorator

A ideia é mostrar como cada padrão ajuda a organizar regras de precificação e desconto de forma flexível e sustentável.

## 1) Factory Method

O padrão Factory Method centraliza a criação de objetos quando a decisão depende da categoria da empresa. Em vez de espalhar ifs e elses pelo código, a criação é delegada para uma fábrica.

Problema:
- criação condicional espalhada por vários pontos do código
- um único lugar decide qual implementação nasce
- o caso de uso pede o quê e deixa de conhecer o como

Solução:
- uma fábrica retorna a política correta para cada tipo de empresa
- o restante do sistema usa a interface comum sem saber os detalhes
- novas categorias podem entrar sem mexer no cliente

In [5]:
from abc import ABC, abstractmethod

class PoliticaDesconto(ABC):
    @abstractmethod
    def calcular(self, valor):
        pass

class StartupPolicy(PoliticaDesconto):
    def calcular(self, valor):
        return valor * 0.10

class PmePolicy(PoliticaDesconto):
    def calcular(self, valor):
        return valor * 0.15

class EnterprisePolicy(PoliticaDesconto):
    def calcular(self, valor):
        return valor * 0.20

class FabricaPoliticaDesconto:
    @staticmethod
    def criar(porte):
        politicas = {
            "startup": StartupPolicy(),
            "pme": PmePolicy(),
            "enterprise": EnterprisePolicy(),
        }
        return politicas.get(porte, StartupPolicy())

for porte in ["startup", "pme", "enterprise"]:
    politica = FabricaPoliticaDesconto.criar(porte)
    valor = 5000
    desconto = politica.calcular(valor)
    print(f"{porte}: desconto = R$ {desconto:.2f}; total = R$ {valor - desconto:.2f}")


startup: desconto = R$ 500.00; total = R$ 4500.00
pme: desconto = R$ 750.00; total = R$ 4250.00
enterprise: desconto = R$ 1000.00; total = R$ 4000.00


### Explicação

O Factory Method centraliza a lógica de criação. Assim, o cliente trabalha com a interface da política, enquanto a fábrica escolhe a implementação correta para cada tipo de empresa. Isso reduz acoplamento e facilita a manutenção.

## 2) Observer

O padrão Observer desacopla a reação da regra principal. Em vez de o código da política conhecer todos os departamentos afetados, ela apenas informa que a regra mudou; os interessados são notificados separadamente.

Problema:
- um método que faz cinco coisas depois da regra principal
- consistência obrigatória fica explícita no caso de uso; reação opcional vira assintente
- no diagrama, a cadeia de mensagens pós-passo principal encurta
- limite: se a ordem das reações importa, Observer é o padrão errado

Solução:
- a política emite evento de mudança de regra
- cada observador reage conforme sua responsabilidade
- o sujeito não precisa saber quem vai receber a notificação

In [6]:
from abc import ABC, abstractmethod

class Observer(ABC):
    @abstractmethod
    def atualizar(self, mensagem):
        pass

class VendasObserver(Observer):
    def atualizar(self, mensagem):
        print(f"[Vendas] {mensagem}")

class FinanceiroObserver(Observer):
    def atualizar(self, mensagem):
        print(f"[Financeiro] {mensagem}")

class DiretoriaObserver(Observer):
    def atualizar(self, mensagem):
        print(f"[Diretoria] {mensagem}")

class PoliticaDesconto:
    def __init__(self):
        self._observadores = []
        self._desconto_atual = 0

    def adicionar_observador(self, observador):
        self._observadores.append(observador)

    def remover_observador(self, observador):
        self._observadores.remove(observador)

    def alterar_desconto(self, novo_desconto):
        self._desconto_atual = novo_desconto
        self.notificar()

    def notificar(self):
        for observador in self._observadores:
            observador.atualizar(
                f"Nova política de desconto definida: {self._desconto_atual:.0%}"
            )

politica = PoliticaDesconto()
politica.adicionar_observador(VendasObserver())
politica.adicionar_observador(FinanceiroObserver())
politica.adicionar_observador(DiretoriaObserver())

politica.alterar_desconto(0.18)


[Vendas] Nova política de desconto definida: 18%
[Financeiro] Nova política de desconto definida: 18%
[Diretoria] Nova política de desconto definida: 18%


### Explicação

O Observer separa a regra principal da reação. Quando a política muda, os observadores recebem a notificação sem que o código do desconto precise conhecer detalhes das áreas que serão impactadas.

## 3) Strategy

O padrão Strategy remove a lógica de decisão espalhada por condicionais e encapsula cada regra de desconto em uma classe separada. Na prática, o sistema troca a estratégia conforme a categoria da empresa sem escrever vários ifs.

Problema:
- calcular_desconto com 90 linhas, crescendo a cada categoria nova
- três times editando a mesma função: 4 conflitos de merge em 2 meses
- nova regra = editar código antigo e arriscar quebrar casos existentes

Solução:
- uma classe por política, todas implementando a mesma interface
- uma função ou classe escolhe a estratégia correta
- a regra fica isolada e pode ser trocada sem mexer no código principal

In [7]:
from abc import ABC, abstractmethod

class PoliticaDesconto(ABC):
    @abstractmethod
    def calcular(self, valor):
        pass

class DescontoStartup(PoliticaDesconto):
    def calcular(self, valor):
        return valor * 0.10

class DescontoPme(PoliticaDesconto):
    def calcular(self, valor):
        return valor * 0.15

class DescontoEnterprise(PoliticaDesconto):
    def calcular(self, valor):
        return valor * 0.20

class EstrategiaDesconto:
    def __init__(self, porte):
        self.porte = porte

    def obter_politica(self):
        politicas = {
            "startup": DescontoStartup(),
            "pme": DescontoPme(),
            "enterprise": DescontoEnterprise(),
        }
        return politicas.get(self.porte, DescontoStartup())

class Pedido:
    def __init__(self, valor, porte):
        self.valor = valor
        self.porte = porte

    def calcular_total(self):
        politica = EstrategiaDesconto(self.porte).obter_politica()
        desconto = politica.calcular(self.valor)
        return self.valor - desconto

for porte in ["startup", "pme", "enterprise"]:
    pedido = Pedido(5000, porte)
    print(f"{porte}: R$ {pedido.calcular_total():.2f}")


startup: R$ 4500.00
pme: R$ 4250.00
enterprise: R$ 4000.00


### Explicação

O Strategy encapsula cada regra de desconto em uma estratégia diferente. Em vez de uma função gigantesca com ifs, o código envia a decisão para um objeto responsável por aquela política. Isso reduz acoplamento, melhora manutenção e facilita evolução sem quebrar o restante do sistema.

## 4) Adapter

O padrão Adapter é usado para adaptar uma API externa a uma interface esperada pelo sistema interno. Neste caso, a empresa usa um provedor externo de regras de desconto, mas a aplicação local trabalha com uma interface própria.

Problema:
- API externa retorna payload em outro formato
- o sistema interno espera um método simples para calcular desconto
- o restante da aplicação não deve conhecer os detalhes do fornecedor

- o resto do sistema não depende de detalhes do fornecedor

Solução:- o adaptador converte a resposta externa para o modelo esperado
- a aplicação continua consumindo uma interface interna simples

In [8]:
class ApiExternaDesconto:
    def obter_regra(self, porte):
        regras = {
            "startup": {"percentual": 5, "codigo": "STP"},
            "pme": {"percentual": 10, "codigo": "PME"},
            "enterprise": {"percentual": 18, "codigo": "ENT"}
        }
        return regras.get(porte, {"percentual": 5, "codigo": "STP"})

class InterfaceInternaDesconto:
    def calcular(self, valor, porte):
        raise NotImplementedError

class AdaptadorApiDesconto(InterfaceInternaDesconto):
    def __init__(self, api_externa):
        self.api_externa = api_externa

    def calcular(self, valor, porte):
        regra = self.api_externa.obter_regra(porte)
        percentual = regra["percentual"] / 100
        return valor * percentual

api_externa = ApiExternaDesconto()
adaptador = AdaptadorApiDesconto(api_externa)
valor = 9000
porte = "pme"
desconto = adaptador.calcular(valor, porte)

print(f"Empresa: {porte}")
print(f"Valor: R$ {valor:.2f}")
print(f"Desconto externo traduzido: R$ {desconto:.2f}")
print(f"Valor final: R$ {valor - desconto:.2f}")


Empresa: pme
Valor: R$ 9000.00
Desconto externo traduzido: R$ 900.00
Valor final: R$ 8100.00


### Explicação

O Adapter traduz a resposta da API externa para a interface do sistema interno. Assim, o restante da aplicação continua usando uma API simples, sem depender dos detalhes de serialização, nomes de campos e formatos do fornecedor externo.

## 5) Decorator

O padrão Decorator adiciona comportamentos extras sem alterar a interface do objeto principal. No contexto de descontos, a regra base continua sendo a mesma, mas ela pode receber camadas de log, cache e retry sem que o cliente saiba disso.

Problema:
- a classe base cresce com responsabilidades extras
- cada nova regra exige alteração no comportamento principal
- o cliente precisa manter lógica de infraestrutura junto ao cálculo

Solução:
- cada camada adiciona uma responsabilidade específica
- o cliente usa a mesma interface sem perceber as camadas extras
- o objeto pode ser composto dinamicamente

In [10]:
from abc import ABC, abstractmethod

class PoliticaDesconto(ABC):
    @abstractmethod
    def calcular(self, valor):
        pass

class DescontoBase(PoliticaDesconto):
    def calcular(self, valor):
        return valor * 0.90

class DecoratorDesconto(PoliticaDesconto):
    def __init__(self, politica):
        self.politica = politica

class LogDesconto(DecoratorDesconto):
    def calcular(self, valor):
        resultado = self.politica.calcular(valor)
        print(f"[Log] Valor calculado: R$ {resultado:.2f}")
        return resultado

class CacheDesconto(DecoratorDesconto):
    def __init__(self, politica):
        super().__init__(politica)
        self._cache = {}

    def calcular(self, valor):
        if valor not in self._cache:
            self._cache[valor] = self.politica.calcular(valor)
        return self._cache[valor]

class RetryDesconto(DecoratorDesconto):
    def calcular(self, valor):
        for tentativa in range(3):
            try:
                return self.politica.calcular(valor)
            except Exception:
                if tentativa == 2:
                    raise RuntimeError("Falha ao calcular o desconto após 3 tentativas.")

valor = 25000
politica = DescontoBase()
politica = LogDesconto(politica)
politica = CacheDesconto(politica)
politica = RetryDesconto(politica)

valor_final = politica.calcular(valor)
print(f"Valor final com decoração: R$ {valor_final:.2f}")


[Log] Valor calculado: R$ 22500.00
Valor final com decoração: R$ 22500.00


### Explicação

O Decorator permite empilhar responsabilidades de forma dinâmica, sem alterar a classe base. Aqui, a política de desconto continua sendo calculada normalmente, mas recebe camadas extras de observabilidade, cache e recuperação de falhas. Isso deixa o código mais flexível e reutilizável.

## Conclusão

Os padrões de projeto ajudam a modelar a política de desconto para empresas de diferentes portes de forma mais organizada:

- Factory Method: cria a regra correta para cada empresa.
- Observer: avisa interessados quando a regra muda.
- Strategy: troca estratégias de cálculo sem alterar o pedido.
- Adapter: integra sistemas antigos com novos formatos.
- Decorator: combina múltiplos descontos de maneira dinâmica.

Esses padrões tornam o código mais reutilizável, escalável e fácil de manter.